# 02_parameter_estimation_ideal

This notebook turns the shared cleaned data into model-ready parameters for the Part 1 benchmark model.

Main outputs:
- facility-level expansion parameters
- ZIP-code demand and supply parameters
- build-option parameters for new facilities
- report summary tables and exported CSV inputs for the benchmark optimization model


In [1]:
# 02_parameter_estimation_ideal

'''This notebook converts the shared cleaned datasets into model-ready inputs for the ideal child care expansion model.

Main tasks:
1. Load shared cleaned datasets
2. Define modeling assumptions for the ideal scenario
3. Construct zipcode-level demand and supply parameters
4. Construct facility-level expansion parameters
5. Export model input tables for the Gurobi ideal model'''

'This notebook converts the shared cleaned datasets into model-ready inputs for the ideal child care expansion model.\n\nMain tasks:\n1. Load shared cleaned datasets\n2. Define modeling assumptions for the ideal scenario\n3. Construct zipcode-level demand and supply parameters\n4. Construct facility-level expansion parameters\n5. Export model input tables for the Gurobi ideal model'

## Setup and shared inputs

The first cells load the cleaned outputs from Notebook 01 and verify that the benchmark parameter-estimation stage has all required inputs available.


In [2]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
#Display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

In [4]:
#Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SHARED_DIR = PROJECT_ROOT / "data" / "processed" / "shared"
IDEAL_DIR = PROJECT_ROOT / "data" / "processed" / "ideal"

IDEAL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SHARED_DIR:", SHARED_DIR)
print("IDEAL_DIR:", IDEAL_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
SHARED_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/shared
IDEAL_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal


In [5]:
#Check required input files
required_files = [
    "clean_childcare_facilities.csv",
    "clean_population.csv",
    "clean_employment.csv",
    "clean_income.csv",
    "master_zipcodes.csv"
]

for f in required_files:
    path = SHARED_DIR / f
    print(f, "exists ->", path.exists())

clean_childcare_facilities.csv exists -> True
clean_population.csv exists -> True
clean_employment.csv exists -> True
clean_income.csv exists -> True
master_zipcodes.csv exists -> True


In [6]:
#Load shared cleaned datasets
fac = pd.read_csv(SHARED_DIR / "clean_childcare_facilities.csv")
pop = pd.read_csv(SHARED_DIR / "clean_population.csv")
emp = pd.read_csv(SHARED_DIR / "clean_employment.csv")
inc = pd.read_csv(SHARED_DIR / "clean_income.csv")
master_zip = pd.read_csv(SHARED_DIR / "master_zipcodes.csv")

In [7]:
#Quick preview
datasets = {
    "facilities": fac,
    "population": pop,
    "employment": emp,
    "income": inc,
    "master_zip": master_zip
}

for name, df in datasets.items():
    print("=" * 100)
    print(name, df.shape)
    print(df.head(3))

facilities (7740, 20)
   facility_id program_type facility_status                           facility_name      city  zipcode school_district_name  infant_capacity  \
0        51349          FDC    Registration                         Valerio, Gladys     Bronx    10474              Bronx 8                0   
1        73544         SACC    Registration                      YMCA OF GREATER NY  New York    10017          Manhattan 2                0   
2       108407         GFDC         License  Almond Tree Group Family Day Care, LLC  Brooklyn    11225          Brooklyn 17                0   

   toddler_capacity  preschool_capacity  school_age_capacity  children_capacity  total_capacity   latitude  longitude  is_active_facility  geo_missing_flag  \
0                 0                   0                    0                  6               6  40.818399 -73.888816                   1             False   
1                 0                   0                   60                  0    

## Assumptions and helper functions

This section records the benchmark modeling assumptions and defines the helper functions used to convert project rules into integer-valued optimization parameters.


In [8]:
#Modeling assumptions for the ideal scenario
ASSUMPTIONS = {
    "active_facility_statuses": ["License", "Registration"],
    "use_children_capacity_as_under5_proxy": True,
    "income_missing_imputation": "citywide_median",
    "employment_missing_imputation": "citywide_median",
    "population_missing_imputation": "treat_as_zero_after_merge",
    "ideal_expansion_cap_option": 2,
    "ideal_expansion_cap_option_1_definition": "x_f <= max(0, min(1.2*n_f, 500 - n_f))",
    "ideal_expansion_cap_option_2_definition": "x_f <= min(1.2*n_f, 500)",
    "chosen_expansion_cap_option": 2,
    "high_demand_rule": "employment_rate >= 0.60 OR average_income <= 60000",
    "desert_threshold_high_demand": "available_slots <= 0.5 * child_population_0_12",
    "desert_threshold_normal_demand": "available_slots <= (1/3) * child_population_0_12",
    "under5_requirement": "available_under5_slots >= (2/3) * pop_0_5",
    "total_requirement_integer_rule": "minimum non-desert integer capacity = floor(threshold) + 1",
    "under5_requirement_integer_rule": "minimum integer under5 capacity = ceil((2/3) * pop_0_5)",
    "cslots_function_type": "tiered_decreasing",
    "cslots_notes": "decreasing per-slot expansion cost to reflect economies of scale"
}

ASSUMPTIONS

{'active_facility_statuses': ['License', 'Registration'],
 'use_children_capacity_as_under5_proxy': True,
 'income_missing_imputation': 'citywide_median',
 'employment_missing_imputation': 'citywide_median',
 'population_missing_imputation': 'treat_as_zero_after_merge',
 'ideal_expansion_cap_option': 2,
 'ideal_expansion_cap_option_1_definition': 'x_f <= max(0, min(1.2*n_f, 500 - n_f))',
 'ideal_expansion_cap_option_2_definition': 'x_f <= min(1.2*n_f, 500)',
 'chosen_expansion_cap_option': 2,
 'high_demand_rule': 'employment_rate >= 0.60 OR average_income <= 60000',
 'desert_threshold_high_demand': 'available_slots <= 0.5 * child_population_0_12',
 'desert_threshold_normal_demand': 'available_slots <= (1/3) * child_population_0_12',
 'under5_requirement': 'available_under5_slots >= (2/3) * pop_0_5',
 'total_requirement_integer_rule': 'minimum non-desert integer capacity = floor(threshold) + 1',
 'under5_requirement_integer_rule': 'minimum integer under5 capacity = ceil((2/3) * pop_0_5)

In [9]:
def cslots_tiered(n):
    """
    Decreasing per-slot cost function for the first expansion block.
    You may change the thresholds in sensitivity analysis.
    """
    if pd.isna(n):
        return np.nan
    if n < 25:
        return 350
    elif n < 75:
        return 300
    else:
        return 250


def total_requirement_not_desert(pop_0_12, high_demand):
    """
    Project rule:
    - high-demand desert if slots <= 1/2 of child population
    - otherwise desert if slots <= 1/3 of child population
    
    To be NOT a desert with integer slots:
    required_total = floor(threshold) + 1
    """
    if pd.isna(pop_0_12):
        return np.nan
    if pop_0_12 <= 0:
        return 0
    
    threshold = 0.5 * pop_0_12 if high_demand == 1 else (1/3) * pop_0_12
    return int(np.floor(threshold) + 1)


def under5_requirement(pop_0_5):
    """
    Project rule:
    under-5 slots must be at least 2/3 of the 0-5 population.
    Since capacity is integer, use ceil.
    """
    if pd.isna(pop_0_5):
        return np.nan
    return int(np.ceil((2/3) * pop_0_5))

In [10]:
#Build options table for new facilities
build_options = pd.DataFrame({
    "size": ["small", "medium", "large"],
    "new_total_capacity": [100, 200, 400],
    "new_under5_capacity_max": [50, 100, 200],
    "fixed_build_cost": [65000, 95000, 115000]
})

build_options

,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost
0,small,100,50,65000
1,medium,200,100,95000
2,large,400,200,115000


## Facility-level supply and expansion parameters

These cells filter the regulated-facility data to the active benchmark universe and compute the Part 1 expansion caps and cost coefficients used by the ideal model.


In [11]:
#Facility table: inspect status and capacities
print("Facility statuses:")
display(fac["facility_status"].value_counts(dropna=False).to_frame("count"))

print("\nActive facility flag counts:")
display(fac["is_active_facility"].value_counts(dropna=False).to_frame("count"))

print("\nSummary of current capacities:")
display(
    fac[["total_capacity", "under5_capacity_current", "children_capacity"]].describe()
)

Facility statuses:


,count
facility_status,
License,5452
Registration,2261
Pending Revocation,16
Pending Revocation and Denial,8
Suspended,3



Active facility flag counts:


,count
is_active_facility,
1,7713
0,27



Summary of current capacities:


,total_capacity,under5_capacity_current,children_capacity
count,7740.000000,7740.000000,7740.000000
mean,41.131137,8.961111,8.961111
std,70.966766,4.807783,4.807783
min,0.000000,0.000000,0.000000
25%,14.000000,6.000000,6.000000
50%,16.000000,12.000000,12.000000
75%,16.000000,12.000000,12.000000
max,890.000000,12.000000,12.000000


In [12]:
#Filter facilities for the ideal model
ideal_fac = fac.copy()

# Keep only active facilities for ideal model
ideal_fac = ideal_fac[ideal_fac["is_active_facility"] == 1].copy()

# Keep facilities with positive total capacity
ideal_fac = ideal_fac[ideal_fac["total_capacity"].fillna(0) > 0].copy()

# Fill under-5 current capacity with 0 if missing
ideal_fac["under5_capacity_current"] = ideal_fac["under5_capacity_current"].fillna(0)

# Basic aliases
ideal_fac["n_f"] = ideal_fac["total_capacity"].astype(float)
ideal_fac["b_f"] = ideal_fac["under5_capacity_current"].astype(float)

print("Ideal facility table shape:", ideal_fac.shape)
display(ideal_fac.head())

Ideal facility table shape: (7712, 22)


,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,is_active_facility,geo_missing_flag,under5_capacity_current,capacity_inconsistency_flag,zero_total_capacity_flag,n_f,b_f
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816,1,False,6,0,0,6.0,6.0
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794,1,False,0,0,0,60.0,0.0
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0,16.0,12.0
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0,16.0,12.0
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572,1,False,12,0,0,12.0,12.0


In [13]:
#Expansion cap choice for the ideal model
EXPANSION_CAP_OPTION = ASSUMPTIONS["chosen_expansion_cap_option"]

if EXPANSION_CAP_OPTION == 1:
    # x_f <= max(0, min(1.2*n_f, 500 - n_f))
    ideal_fac["U_f"] = np.maximum(
        0,
        np.minimum(1.2 * ideal_fac["n_f"], 500 - ideal_fac["n_f"])
    )
elif EXPANSION_CAP_OPTION == 2:
    # x_f <= min(1.2*n_f, 500)
    ideal_fac["U_f"] = np.minimum(1.2 * ideal_fac["n_f"], 500)
else:
    raise ValueError("EXPANSION_CAP_OPTION must be 1 or 2")

ideal_fac[["facility_id", "zipcode", "n_f", "U_f"]].head()

,facility_id,zipcode,n_f,U_f
0,51349,10474,6.0,7.2
1,73544,10017,60.0,72.0
2,108407,11225,16.0,19.2
3,111953,11226,16.0,19.2
4,126425,10465,12.0,14.4


In [14]:
#Expansion cost parameters
ideal_fac["cslot_f"] = ideal_fac["n_f"].apply(cslots_tiered)

# Cost coefficient for the block beyond 100% of existing capacity
# From project statement: (20000 + 200*n_f) * x_f / n_f
# => coefficient on x_f is (20000 + 200*n_f) / n_f
ideal_fac["alpha_big_f"] = (20000 + 200 * ideal_fac["n_f"]) / ideal_fac["n_f"]

# Decompose expansion into:
# x1_f: first block up to n_f
# x2_f: additional block beyond n_f
ideal_fac["x1_cap"] = np.minimum(ideal_fac["n_f"], ideal_fac["U_f"])
ideal_fac["x2_cap"] = np.maximum(0, ideal_fac["U_f"] - ideal_fac["n_f"])

display(
    ideal_fac[[
        "facility_id", "zipcode", "n_f", "U_f", "cslot_f",
        "alpha_big_f", "x1_cap", "x2_cap"
    ]].head()
)

,facility_id,zipcode,n_f,U_f,cslot_f,alpha_big_f,x1_cap,x2_cap
0,51349,10474,6.0,7.2,350,3533.333333,6.0,1.2
1,73544,10017,60.0,72.0,300,533.333333,60.0,12.0
2,108407,11225,16.0,19.2,350,1450.000000,16.0,3.2
3,111953,11226,16.0,19.2,350,1450.000000,16.0,3.2
4,126425,10465,12.0,14.4,350,1866.666667,12.0,2.4


In [15]:
#Facility parameter sanity checks
print("Any negative U_f?", (ideal_fac["U_f"] < 0).any())
print("Any negative x1_cap?", (ideal_fac["x1_cap"] < 0).any())
print("Any negative x2_cap?", (ideal_fac["x2_cap"] < 0).any())
print("Any missing cslot_f?", ideal_fac["cslot_f"].isna().sum())
print("Any missing alpha_big_f?", ideal_fac["alpha_big_f"].isna().sum())

Any negative U_f? False
Any negative x1_cap? False
Any negative x2_cap? False
Any missing cslot_f? 0
Any missing alpha_big_f? 0


In [16]:
#Keep only columns needed for the optimization model
facility_expansion_params_ideal = ideal_fac[[
    "facility_id",
    "facility_name",
    "zipcode",
    "program_type",
    "facility_status",
    "n_f",
    "b_f",
    "U_f",
    "cslot_f",
    "alpha_big_f",
    "x1_cap",
    "x2_cap"
]].copy()

facility_expansion_params_ideal = facility_expansion_params_ideal.sort_values(
    by=["zipcode", "facility_id"]
).reset_index(drop=True)

facility_expansion_params_ideal.head()

,facility_id,facility_name,zipcode,program_type,facility_status,n_f,b_f,U_f,cslot_f,alpha_big_f,x1_cap,x2_cap
0,229433,YMCA of Greater NY @Virtual Y PS 33,10001,SACC,Registration,88.0,0.0,105.6,250,427.272727,88.0,17.6
1,292419,The Hudson Guild @26th Street,10001,SACC,Registration,79.0,0.0,94.8,250,453.164557,79.0,15.8
2,350076,GRAMMAS HANDS,10001,FDC,Registration,8.0,6.0,9.6,350,2700.000000,8.0,1.6
3,661697,Chelsea Little Angels Day Care,10001,GFDC,License,16.0,12.0,19.2,350,1450.000000,16.0,3.2
4,827488,"Davis, Milinaire",10001,FDC,Registration,8.0,6.0,9.6,350,2700.000000,8.0,1.6


## ZIP-code demand and baseline supply

This section builds the benchmark ZIP-code table, imputes missing demographic covariates, and computes the total-capacity and under-5 requirements used in the optimization model.


In [17]:
#Start zipcode-level parameter table
zip_base = master_zip[["zipcode"]].copy()

zip_base = zip_base.merge(
    pop[["zipcode", "pop_0_5", "pop_6_12", "child_pop_0_12"]],
    on="zipcode",
    how="left"
)

zip_base = zip_base.merge(
    emp[["zipcode", "employment_rate"]],
    on="zipcode",
    how="left"
)

zip_base = zip_base.merge(
    inc[["zipcode", "average_income"]],
    on="zipcode",
    how="left"
)

print("zip_base shape:", zip_base.shape)
display(zip_base.head())

zip_base shape: (311, 6)


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645
3,10004,433.0,262.0,695.0,0.506661,132004.310345
4,10005,484.0,318.0,802.0,0.665833,121437.713311


In [18]:
#Impute missing employment and income
employment_median = zip_base["employment_rate"].median(skipna=True)
income_median = zip_base["average_income"].median(skipna=True)

zip_base["employment_missing_flag"] = zip_base["employment_rate"].isna().astype(int)
zip_base["income_missing_flag"] = zip_base["average_income"].isna().astype(int)

zip_base["employment_rate"] = zip_base["employment_rate"].fillna(employment_median)
zip_base["average_income"] = zip_base["average_income"].fillna(income_median)

print("employment median used for imputation:", employment_median)
print("income median used for imputation:", income_median)

employment median used for imputation: 0.4766521247374374
income median used for imputation: 61197.28128267689


In [19]:
#Handle population missingness
for col in ["pop_0_5", "pop_6_12", "child_pop_0_12"]:
    zip_base[f"{col}_missing_flag"] = zip_base[col].isna().astype(int)
    zip_base[col] = zip_base[col].fillna(0)

display(
    zip_base[[
        "zipcode", "pop_0_5", "pop_6_12", "child_pop_0_12",
        "employment_rate", "average_income"
    ]].head()
)

,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645
3,10004,433.0,262.0,695.0,0.506661,132004.310345
4,10005,484.0,318.0,802.0,0.665833,121437.713311


In [20]:
#High-demand indicator
zip_base["high_demand"] = (
    (zip_base["employment_rate"] >= 0.60) |
    (zip_base["average_income"] <= 60000)
).astype(int)

display(zip_base["high_demand"].value_counts().to_frame("count"))

,count
high_demand,
0,215
1,96


In [21]:
#Compute total and under-5 requirements
zip_base["total_threshold_rule"] = np.where(
    zip_base["high_demand"] == 1,
    0.5 * zip_base["child_pop_0_12"],
    (1/3) * zip_base["child_pop_0_12"]
)

zip_base["req_total"] = zip_base.apply(
    lambda row: total_requirement_not_desert(row["child_pop_0_12"], row["high_demand"]),
    axis=1
)

zip_base["req_under5"] = zip_base["pop_0_5"].apply(under5_requirement)

display(
    zip_base[[
        "zipcode", "child_pop_0_12", "high_demand",
        "total_threshold_rule", "req_total", "pop_0_5", "req_under5"
    ]].head()
)

,zipcode,child_pop_0_12,high_demand,total_threshold_rule,req_total,pop_0_5,req_under5
0,10001,1999.0,0,666.333333,667,744.0,496
1,10002,6787.0,1,3393.500000,3394,2142.0,1428
2,10003,2950.0,0,983.333333,984,1440.0,960
3,10004,695.0,0,231.666667,232,433.0,289
4,10005,802.0,1,401.000000,402,484.0,323


In [22]:
#Build current zipcode supply from active facilities
zipcode_supply = (
    ideal_fac.groupby("zipcode")
    .agg(
        existing_total=("n_f", "sum"),
        existing_under5=("b_f", "sum"),
        n_active_facilities=("facility_id", "count")
    )
    .reset_index()
)

display(zipcode_supply.head())

,zipcode,existing_total,existing_under5,n_active_facilities
0,10001,609.0,24.0,9
1,10002,4724.0,198.0,56
2,10003,1995.0,0.0,7
3,10004,263.0,0.0,2
4,10005,39.0,0.0,1


In [23]:
#Merge supply into zipcode parameter table
zip_base = zip_base.merge(zipcode_supply, on="zipcode", how="left")

for col in ["existing_total", "existing_under5", "n_active_facilities"]:
    zip_base[col] = zip_base[col].fillna(0)

display(
    zip_base[[
        "zipcode", "existing_total", "existing_under5", "n_active_facilities"
    ]].head()
)

,zipcode,existing_total,existing_under5,n_active_facilities
0,10001,609.0,24.0,9.0
1,10002,4724.0,198.0,56.0
2,10003,1995.0,0.0,7.0
3,10004,263.0,0.0,2.0
4,10005,39.0,0.0,1.0


## Baseline shortage diagnostics

Once requirements and existing supply are in the same table, the notebook computes pre-solve shortage metrics and benchmark desert indicators for reporting.


In [24]:
#Compute shortages and baseline desert indicators
zip_base["gap_total"] = np.maximum(0, zip_base["req_total"] - zip_base["existing_total"])
zip_base["gap_under5"] = np.maximum(0, zip_base["req_under5"] - zip_base["existing_under5"])

zip_base["is_desert_total_before"] = np.where(
    zip_base["child_pop_0_12"] > 0,
    zip_base["existing_total"] <= zip_base["total_threshold_rule"],
    0
).astype(int)

zip_base["is_under5_short_before"] = (
    zip_base["existing_under5"] < zip_base["req_under5"]
).astype(int)

zip_base["needs_intervention_before"] = (
    (zip_base["gap_total"] > 0) | (zip_base["gap_under5"] > 0)
).astype(int)

display(
    zip_base[[
        "zipcode",
        "existing_total", "req_total", "gap_total",
        "existing_under5", "req_under5", "gap_under5",
        "is_desert_total_before", "is_under5_short_before",
        "needs_intervention_before"
    ]].head(10)
)

,zipcode,existing_total,req_total,gap_total,existing_under5,req_under5,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before
0,10001,609.0,667,58.0,24.0,496,472.0,1,1,1
1,10002,4724.0,3394,0.0,198.0,1428,1230.0,0,1,1
2,10003,1995.0,984,0.0,0.0,960,960.0,0,1,1
3,10004,263.0,232,0.0,0.0,289,289.0,0,1,1
4,10005,39.0,402,363.0,0.0,323,323.0,1,1,1
5,10006,156.0,131,0.0,0.0,86,86.0,0,1,1
6,10007,284.0,401,117.0,0.0,404,404.0,1,1,1
7,10008,0.0,0,0.0,0.0,0,0.0,0,0,0
8,10009,1784.0,1491,0.0,106.0,1264,1158.0,0,1,1
9,10010,234.0,1163,929.0,0.0,948,948.0,1,1,1


In [25]:
#Final zipcode parameter table
zipcode_demand_supply_ideal = zip_base[[
    "zipcode",
    "pop_0_5",
    "pop_6_12",
    "child_pop_0_12",
    "employment_rate",
    "average_income",
    "employment_missing_flag",
    "income_missing_flag",
    "pop_0_5_missing_flag",
    "pop_6_12_missing_flag",
    "child_pop_0_12_missing_flag",
    "high_demand",
    "total_threshold_rule",
    "req_total",
    "req_under5",
    "existing_total",
    "existing_under5",
    "n_active_facilities",
    "gap_total",
    "gap_under5",
    "is_desert_total_before",
    "is_under5_short_before",
    "needs_intervention_before"
]].copy()

zipcode_demand_supply_ideal = zipcode_demand_supply_ideal.sort_values("zipcode").reset_index(drop=True)

zipcode_demand_supply_ideal.head()

,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1


## Summary tables and exports

The last cells assemble compact reporting tables and export the benchmark parameter files consumed directly by the Part 1 optimization notebook.


In [26]:
#Summary table for the report
summary_rows = []

summary_rows.append({
    "metric": "number_of_zipcodes",
    "value": zipcode_demand_supply_ideal["zipcode"].nunique()
})

summary_rows.append({
    "metric": "number_of_active_facilities",
    "value": facility_expansion_params_ideal["facility_id"].nunique()
})

summary_rows.append({
    "metric": "total_existing_capacity_citywide",
    "value": zipcode_demand_supply_ideal["existing_total"].sum()
})

summary_rows.append({
    "metric": "total_existing_under5_capacity_citywide",
    "value": zipcode_demand_supply_ideal["existing_under5"].sum()
})

summary_rows.append({
    "metric": "zipcodes_desert_before_total_rule",
    "value": zipcode_demand_supply_ideal["is_desert_total_before"].sum()
})

summary_rows.append({
    "metric": "zipcodes_under5_short_before",
    "value": zipcode_demand_supply_ideal["is_under5_short_before"].sum()
})

summary_rows.append({
    "metric": "zipcodes_needing_intervention_before",
    "value": zipcode_demand_supply_ideal["needs_intervention_before"].sum()
})

summary_rows.append({
    "metric": "citywide_total_gap",
    "value": zipcode_demand_supply_ideal["gap_total"].sum()
})

summary_rows.append({
    "metric": "citywide_under5_gap",
    "value": zipcode_demand_supply_ideal["gap_under5"].sum()
})

summary_rows.append({
    "metric": "employment_rate_median_imputation_value",
    "value": employment_median
})

summary_rows.append({
    "metric": "average_income_median_imputation_value",
    "value": income_median
})

summary_rows.append({
    "metric": "chosen_expansion_cap_option",
    "value": EXPANSION_CAP_OPTION
})

summary_rows.append({
    "metric": "avg_facility_expansion_cap_U_f",
    "value": facility_expansion_params_ideal["U_f"].mean()
})

summary_rows.append({
    "metric": "max_facility_expansion_cap_U_f",
    "value": facility_expansion_params_ideal["U_f"].max()
})

model_inputs_summary_ideal = pd.DataFrame(summary_rows)

model_inputs_summary_ideal

,metric,value
0,number_of_zipcodes,311.000000
1,number_of_active_facilities,7712.000000
2,total_existing_capacity_citywide,317501.000000
3,total_existing_under5_capacity_citywide,69153.000000
4,zipcodes_desert_before_total_rule,162.000000
5,zipcodes_under5_short_before,179.000000
6,zipcodes_needing_intervention_before,180.000000
7,citywide_total_gap,236460.000000
8,citywide_under5_gap,277721.000000
9,employment_rate_median_imputation_value,0.476652


In [27]:
#top shortage zipcodes preview
print("Top 15 zipcodes by total gap:")
display(
    zipcode_demand_supply_ideal.sort_values("gap_total", ascending=False)[[
        "zipcode", "gap_total", "gap_under5", "high_demand",
        "existing_total", "req_total", "existing_under5", "req_under5"
    ]].head(15)
)

print("\nTop 15 zipcodes by under-5 gap:")
display(
    zipcode_demand_supply_ideal.sort_values("gap_under5", ascending=False)[[
        "zipcode", "gap_under5", "gap_total", "high_demand",
        "existing_total", "req_total", "existing_under5", "req_under5"
    ]].head(15)
)

Top 15 zipcodes by total gap:


,zipcode,gap_total,gap_under5,high_demand,existing_total,req_total,existing_under5,req_under5
213,11219,10779.0,7072.0,1,2595.0,13374,1012.0,8084
258,11368,6564.0,4761.0,1,3057.0,9621,502.0,5263
217,11223,6293.0,4284.0,1,1448.0,7741,144.0,4428
214,11220,6093.0,3459.0,1,2251.0,8344,396.0,3855
198,11204,5491.0,4374.0,1,3192.0,8683,432.0,4806
200,11206,5385.0,4444.0,1,3233.0,8618,358.0,4802
223,11230,4979.0,4967.0,0,824.0,5803,312.0,5279
202,11208,4760.0,4300.0,1,5054.0,9814,1218.0,5518
305,11691,4432.0,3307.0,1,2880.0,7312,725.0,4032
271,11385,4316.0,3408.0,1,2974.0,7290,610.0,4018



Top 15 zipcodes by under-5 gap:


,zipcode,gap_under5,gap_total,high_demand,existing_total,req_total,existing_under5,req_under5
213,11219,7072.0,10779.0,1,2595.0,13374,1012.0,8084
223,11230,4967.0,4979.0,0,824.0,5803,312.0,5279
258,11368,4761.0,6564.0,1,3057.0,9621,502.0,5263
200,11206,4444.0,5385.0,1,3233.0,8618,358.0,4802
198,11204,4374.0,5491.0,1,3192.0,8683,432.0,4806
202,11208,4300.0,4760.0,1,5054.0,9814,1218.0,5518
217,11223,4284.0,6293.0,1,1448.0,7741,144.0,4428
214,11220,3459.0,6093.0,1,2251.0,8344,396.0,3855
263,11373,3414.0,3918.0,1,3358.0,7276,246.0,3660
271,11385,3408.0,4316.0,1,2974.0,7290,610.0,4018


In [28]:
#Save assumptions table
assumptions_ideal = pd.DataFrame(
    [{"assumption": k, "value": str(v)} for k, v in ASSUMPTIONS.items()]
)

assumptions_ideal

,assumption,value
0,active_facility_statuses,"['License', 'Registration']"
1,use_children_capacity_as_under5_proxy,True
2,income_missing_imputation,citywide_median
3,employment_missing_imputation,citywide_median
4,population_missing_imputation,treat_as_zero_after_merge
5,ideal_expansion_cap_option,2
6,ideal_expansion_cap_option_1_definition,"x_f <= max(0, min(1.2*n_f, 500 - n_f))"
7,ideal_expansion_cap_option_2_definition,"x_f <= min(1.2*n_f, 500)"
8,chosen_expansion_cap_option,2
9,high_demand_rule,employment_rate >= 0.60 OR average_income <= 6...


In [29]:
#Export all ideal parameter files
facility_expansion_params_ideal.to_csv(
    IDEAL_DIR / "facility_expansion_params_ideal.csv",
    index=False
)

zipcode_demand_supply_ideal.to_csv(
    IDEAL_DIR / "zipcode_demand_supply_ideal.csv",
    index=False
)

build_options.to_csv(
    IDEAL_DIR / "build_options_ideal.csv",
    index=False
)

assumptions_ideal.to_csv(
    IDEAL_DIR / "assumptions_ideal.csv",
    index=False
)

model_inputs_summary_ideal.to_csv(
    IDEAL_DIR / "model_inputs_summary_ideal.csv",
    index=False
)

# Optional: also save raw assumptions as json
with open(IDEAL_DIR / "assumptions_ideal.json", "w", encoding="utf-8") as f:
    json.dump(ASSUMPTIONS, f, ensure_ascii=False, indent=2)

print("Ideal parameter files saved to:", IDEAL_DIR)

Ideal parameter files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal
